In [0]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import IntegerType, TimestampType
from modules.data.bulk_read import get_merged_df

In [0]:
# Update the stations lookup table with the latests records. SCD type 2 strategy is applied by closing any stations whose attributes changed
# 1. Get a Dataframe with records from all cities, from bronze layer.
# 2. Select, rename, cast, and add audit columns (valid_from, valid_to).
# 3. Load SCD type 2 lookup Delta table.
# 4. Close active stations whose attributes changed.
# 5. Append new current versions + new stations with no historical row in target. 
df = get_merged_df(spark=spark, catalog="bikes", schema="01_bronze", substring="station_lookup_raw")

In [0]:
df = df.select(
    col("city"),
    col("station_id"),
    col("name"),
    col("short_name"),
    col("lon").alias("longitude"),
    col("lat").alias("latitude"),
    col("region_id").cast(IntegerType()),
    col("capacity"),
    current_timestamp().alias("valid_from"),
    lit(None).cast(TimestampType()).alias('valid_to')
)

In [0]:
end_timestamp = datetime.now()
dt = DeltaTable.forName(spark, 'bikes.02_silver.stations_cleansed')

In [0]:
actives = dt.toDF().filter(col("valid_to").isNull())

changed = (
    df.alias("s")
      .join(actives.alias("t"), on="station_id", how="inner")
      .where(
          (col("t.name") != col("s.name")) |
          (col("t.short_name") != col("s.short_name")) |
          (col("t.longitude") != col("s.longitude")) |
          (col("t.latitude") != col("s.latitude")) |
          (col("t.region_id") != col("s.region_id"))
      )
      .select("station_id")
)
changed_ids = [r["station_id"] for r in changed.collect()]

if changed_ids:
    dt.update(
        condition = col("valid_to").isNull() & col("station_id").isin(changed_ids),
        set = {"valid_to": lit(end_timestamp).cast(TimestampType())}
    )


In [0]:
existing_ids = [r["station_id"] for r in dt.toDF().select("station_id").distinct().collect()]

to_insert = df.filter(
    col("station_id").isin(changed_ids) | (~col("station_id").isin(existing_ids))
)
to_insert.write.format("delta").mode("append").saveAsTable("bikes.02_silver.stations_cleansed")